- Links uteis:

1 - https://theairlab.org/alfa-dataset/ (contém informações sobre o status do vôo.)

# Explorando os dados


In [1]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt


# Parâmetros

In [2]:
DATA_FILE = '../data/01_raw/processed/carbonZ_2018-07-18-15-53-31_2_engine_failure'

# Funções

In [3]:

def plot_generic_df(df, titulo_grafico="Análise de Dados"):
    # 1. Tratamento do Tempo (Eixo X)
    if '%time' in df.columns:
        t_zero = df['%time'].iloc[0]
        tempo_segundos = (df['%time'] - t_zero) / 1e9
    else:
        print("Coluna %time não encontrada. Usando índice como tempo.")
        tempo_segundos = df.index

    # 2. Filtrar apenas colunas numéricas (e excluir metadados de tempo/sequência)
    cols_numericas = df.select_dtypes(include=['number']).columns.tolist()
    # Ignoramos colunas que são apenas contadores ou carimbos de tempo para o plot ficar limpo
    blacklist = ['%time', 'field.header.stamp', 'field.header.seq']
    cols_para_plotar = [c for c in cols_numericas if c not in blacklist]

    # 3. Configurar Subplots dinamicamente
    # Vamos agrupar as colunas para não criar 20 subplots (ex: máximo de 3 colunas por gráfico)
    num_cols = len(cols_para_plotar)
    if num_cols == 0:
        print("Nenhuma coluna numérica encontrada para plotar.")
        return

    # Criamos subplots: um para cada 3 variáveis (ou conforme sua preferência)
    v_limit = 4 # variáveis por subplot
    n_subplots = (num_cols + v_limit - 1) // v_limit
    
    fig, axes = plt.subplots(n_subplots, 1, figsize=(12, 4 * n_subplots), sharex=True)
    
    # Se houver apenas 1 subplot, o matplotlib não retorna uma lista, então ajustamos:
    if n_subplots == 1: axes = [axes]

    # 4. Plotagem em massa
    for i in range(n_subplots):
        start = i * v_limit
        end = start + v_limit
        batch_cols = cols_para_plotar[start:end]
        
        for col in batch_cols:
            # Pegamos apenas o final do nome da coluna (ex: 'linear.x') para a legenda
            label_curto = col.split('.')[-1]
            axes[i].plot(tempo_segundos, df[col], label=label_curto)
        
        axes[i].legend(loc='upper right', fontsize='small')
        axes[i].grid(True, alpha=0.3)
        axes[i].set_ylabel("Valores")

    axes[-1].set_xlabel("Tempo (segundos)")
    plt.suptitle(titulo_grafico, fontsize=14)
    plt.tight_layout(rect=[0, 0.03, 1, 0.97])
    plt.show()

# Exemplo de uso para o seu primeiro DF:
# plot_generic_df(df_primeiro, titulo_grafico=primeira_chave)

# Abrindo todos os arquivos e salvando em um dicionário de dfs



In [4]:
pasta = Path(DATA_FILE)

caminho_datasets = list(pasta.glob('*.csv'))

#for f in caminho_datasets:
#    print(f)


In [5]:
dfs = {f.stem: pd.read_csv(f) for f in pasta.glob('*.csv')}


ParserError: Error tokenizing data. C error: Expected 16 fields in line 4, saw 18


In [6]:
dfs.keys()


NameError: name 'dfs' is not defined

In [ ]:
len(dfs.keys())


São 34 dataframes para se explorar. 

# Dataframe 1 

In [ ]:
list(dfs.keys())[0]

In [ ]:
i = 0

primeira_chave = list(dfs.keys())[i]
df1 = dfs[primeira_chave]


In [ ]:
df1

In [ ]:
df1.columns

In [ ]:
df1.shape

## Interpretando as Colunas

- %time e field.header.stamp: São os marcadores de tempo. O %time geralmente é o tempo de recepção do sistema, enquanto o stamp é quando o sensor gerou o dado. 

- field.twist.linear.x, y, z: Representam a velocidade linear (em m/s).

- linear.z: Esta é a sua coluna de ouro para falha de motor. Em um voo nivelado, ela deve estar próxima de 0. Se o motor falha, você verá um valor negativo persistente (descida), independentemente do que o piloto ordene.

- linear.x e linear.y: Velocidade de deslocamento horizontal (Norte/Sul, Leste/Oeste).

- field.twist.angular.x, y, z: Representam a velocidade de rotação (taxas de giro em rad/s) em torno dos eixos da aeronave.

No ALFA, se o motor falha em um avião de asa fixa (como o CarbonZ), ele perde sustentação e pode começar a girar (stall). Um aumento repentino em angular.x (roll) ou angular.y (pitch) sem comando prévio indica instabilidade causada pela falha.

In [ ]:
plot_generic_df(df1, titulo_grafico=primeira_chave)

O que está acontecendo provavelmente é um destes dois cenários:

1. O "Voo Fantasma" (Tópico Inativo)

No sistema ROS, quando um sensor não está enviando dados ou o drone ainda está no chão/estático, o log registra apenas zeros ou valores constantes de inicialização. Se as velocidades X e Y oscilam minimamente em torno de zero e Z está parado, esse log pode ser apenas o período de "pre-arm" (o drone parado na pista esperando o comando de decolagem).

2. Velocidade vs. Posição Global

O arquivo que você abriu é o global_position-raw-gps_vel. O GPS às vezes demora para "travar" o sinal (lock) ou só começa a reportar variações significativas após o movimento real começar.

# Dataframe 2

In [ ]:
i = 1
list(dfs.keys())[i]

In [ ]:
primeira_chave = list(dfs.keys())[i]
df2 = dfs[primeira_chave]


In [ ]:
df2

In [ ]:
# 2. Renomeando a coluna para algo sugestivo
df2 = df2.rename(columns={'field.data': 'engine_status'})

In [ ]:
df2.columns

In [ ]:
df2['engine_status'].unique()

## Interpretando as Colunas

- %time e field.header.stamp: São os marcadores de tempo. O %time geralmente é o tempo de recepção do sistema, enquanto o stamp é quando o sensor gerou o dado. 

- engine_status: Indica se a aeronave está saudável ou com falha. De acordo com o Link [1](https://theairlab.org/alfa-dataset/), os dados deste vôo possuíram 16 segundos de voo após a falha, logo, pelo gráfico abaixo o valor 1 indica falha.

In [ ]:
plot_generic_df(df2, titulo_grafico=primeira_chave)

In [ ]:
# O merge_asof alinha o status ao tempo do GPS sem criar linhas extras com NaN
df_merge1 = pd.merge_asof(df1, df2, on='%time', direction='backward')

# Preenchendo os valores ausentes de status com 0 (Normalidade)
df_merge1['engine_status'] = df_merge1['engine_status'].fillna(0).astype(int)

print(df_merge1['engine_status'].value_counts())

In [ ]:
df_merge1

In [ ]:
plot_generic_df(df_merge1, titulo_grafico=primeira_chave)

OBS:

A falha ocorre no final do voo. Os padrões são sutís, mas note a diferença na oscilação da velocidade no eixo y e x.

# Dataframe 3

In [ ]:
i = 2
list(dfs.keys())[i]

In [ ]:
chave = list(dfs.keys())[i]
df3 = dfs[chave]


In [ ]:
df3

In [ ]:
df3.columns

## Interpretando as Colunas

- %time e field.header.stamp: São os marcadores de tempo. O %time geralmente é o tempo de recepção do sistema, enquanto o stamp é quando o sensor gerou o dado. 

- field.rssi: Signal Strength - Força do sinal do rádio (0 a 255). Se cair muito, houve perda de link.

1. O que são as colunas field.channels?

Em aeromodelismo e drones, os canais de rádio (RC) transmitem as intenções do piloto ou do piloto automático. Para um avião de asa fixa (Carbon-Z), a convenção padrão do PX4/ArduPilot costuma ser:


- Canal 0 (Aileron): Inclinação lateral (Roll).

- Canal 1 (Elevator/Profundor): Subir ou descer o nariz (Pitch).

- Canal 2 (Throttle/Acelerador): É o comando de potência para o motor. Valores geralmente entre 1000 (parado) e 2000 (potência máxima).

- Canal 3 (Rudder/Leme): Controla o leme de direção (Yaw).

- Canais 4-15: Geralmente chaves auxiliares (modos de voo, ativação de trem de pouso, ou o interruptor usado para "injetar" a falha).

2. Por que os valores variam pouco?

Os valores desses canais geralmente variam entre 1000 e 2000 (representando o tempo de pulso PWM em microssegundos), onde 1500 é o centro.

- Se o valor está em 1100, o canal está no mínimo.

- Se está em 1900, está no máximo.

Se as variações são pequenas, o piloto estava mantendo um voo estável.

3. A utilidade para o modelo Machine Learning (Semi-supervisionado). Imagine o seguinte cenário:

- O Canal 2 (Acelerador) está em 1800 (Piloto pedindo potência máxima).

- A Velocidade Z (do GPS) começa a cair.

- Conclusão do modelo: "O comando diz para subir, mas o avião está descendo. Isso é uma anomalia!"

4. O campo field.rssi

O RSSI (Received Signal Strength Indicator) mede a força do sinal de rádio entre o controle e o avião. Se o valor for constante e alto, significa que não houve perda de link de rádio. Pode ignorar para a detecção de falha de motor, mas é bom saber que ele está lá caso o avião caísse por perda de sinal.

In [ ]:
# Dicionário de tradução baseado no padrão PX4 para Asa Fixa
mapeamento_canais = {
    'field.channels0': 'rc_aileron',  # Roll
    'field.channels1': 'rc_elevator', # Pitch
    'field.channels2': 'rc_throttle', # Motor (Acelerador)
    'field.channels3': 'rc_rudder',   # Yaw
    'field.rssi': 'rc_signal_strength'
}

# Renomear apenas as que mapeamos e manter o %time
df3 = df3.rename(columns=mapeamento_canais)


In [ ]:
plot_generic_df(df3, titulo_grafico=primeira_chave)

Os canais 4 a 15 geralmente são ruído ou chaves estáticas, então vamos descartá-los para manter o DF limpo.

In [ ]:
# Dicionário de tradução baseado no padrão PX4 para Asa Fixa
mapeamento_canais = {
    'field.channels0': 'rc_aileron',  # Roll
    'field.channels1': 'rc_elevator', # Pitch
    'field.channels2': 'rc_throttle', # Motor (Acelerador)
    'field.channels3': 'rc_rudder',   # Yaw
    'field.rssi': 'rc_signal_strength'
}

# Renomear apenas as que mapeamos e manter o %time
df3 = df3.rename(columns=mapeamento_canais)

# Selecionar apenas as colunas úteis para o merge
colunas_uteis = ['%time'] + list(mapeamento_canais.values())
df3 = df3[colunas_uteis]

In [ ]:
df3

In [ ]:
# Garantir que ambos estão ordenados pelo tempo
df_merge1 = df_merge1.sort_values('%time')
df3 = df3.sort_values('%time')

# Unir os comandos de rádio ao dataframe principal
df_merge2 = pd.merge_asof(df_merge1, df3, on='%time', direction='backward')

# Preencher possíveis NaNs iniciais nos comandos de rádio
df_merge2[list(mapeamento_canais.values())] = df_merge2[list(mapeamento_canais.values())].ffill().fillna(1500) # 1500 é o neutro PWM

print("Colunas agora no DF final:", df_merge2.columns.tolist())

In [ ]:
df_merge2

# Dataframe 4

In [ ]:
i = 3
list(dfs.keys())[i]

In [ ]:
chave = list(dfs.keys())[i]
df4 = dfs[chave]


In [ ]:
df4

In [ ]:
df4.columns

## Interpretando as Colunas

- %time e field.header.stamp: São os marcadores de tempo. O %time geralmente é o tempo de recepção do sistema, enquanto o stamp é quando o sensor gerou o dado. 

1. O que são as colunas field.channels?

Em aeromodelismo e drones, os canais de rádio (RC) transmitem as intenções do piloto ou do piloto automático. Para um avião de asa fixa (Carbon-Z), a convenção padrão do PX4/ArduPilot costuma ser:


O que significam essas colunas?

Elas seguem o protocolo de comunicação MAVLink v2:

- field.sysid e field.compid: ID do sistema (avião) e do componente (autopiloto). Servem para identificar quem enviou a mensagem na rede.

- field.msgid: Esta é a coluna mais importante aqui. Cada número corresponde a um tipo de mensagem (ex: 1 é SYS_STATUS, 30 é ATTITUDE).

- field.payload64x: São os dados brutos (em bytes ou representação decimal) que ainda não foram processados. Sem um decodificador, esses valores parecem "lixo" numérico.

- field.checksum: Um código de verificação para garantir que o pacote não chegou corrompido.

- field.magic: O caractere de início de cada pacote MAVLink (geralmente o valor 253 para MAVLink v2).

2. As colunas acima não são relevantes para o caso de análise de falhas de motor, mas podem ser úteis para avaliar se houve algum ataque ou violação da segurança do vôo. Abaixo estão algumas indicações de como o dado pode ser usado com essa finalidade:

2.1. Detecção de Injeção de Pacotes (Spoofing/Injection)

Se um atacante tentar injetar comandos falsos no drone, ele teria que gerar pacotes MAVLink. Sua colega pode analisar:

- field.seq (Sequence): Os pacotes MAVLink devem seguir uma sequência numérica estrita (0, 1, 2... 255). Se houver saltos bruscos ou duplicatas frequentes, pode indicar que um atacante está tentando "atropelar" o fluxo de dados legítimo.

- field.sysid e field.compid: Se aparecer um ID de sistema que não deveria estar ali, é um sinal claro de um dispositivo intruso na rede de telemetria.

2.2.  Ataques de Man-in-the-Middle ou Corrupção

- field.checksum: Se houver uma alta taxa de pacotes onde o checksum falha (o dado chegou diferente do que foi enviado), isso pode indicar um ataque de interferência proposital (Jamming) ou tentativa de modificar pacotes em trânsito.

- field.framing_status: Indica se o "envelope" do pacote está malformado. Atacantes novatos costumam gerar pacotes que violam o protocolo, o que seria registrado aqui.

2.3. Análise de Payload (Anomalias de Protocolo)

Pode-se Machine Learning para aprender o padrão "normal" dos bytes nos field.payload64x.

- Ataques de Fuzzing (enviar dados aleatórios para travar o sistema) seriam detectados como mudanças drásticas na entropia ou na distribuição desses valores de payload brutos.

In [ ]:
plot_generic_df(df4, titulo_grafico=primeira_chave)

## Observação:

- Não sei o que fazer com as colunas acima, mas talvez sejam úteis para o projeto de segurança Cibernética (Elaine).

In [ ]:
# Garantir que ambos estão ordenados pelo tempo
df_merge2 = df_merge2.sort_values('%time')
df4 = df4.sort_values('%time')

# Unir os comandos de rádio ao dataframe principal
df_merge3 = pd.merge_asof(df_merge2, df4, on='%time', direction='backward')


print("Colunas agora no DF final:", df_merge3.columns.tolist())

In [ ]:
df_merge3

# Dataframe 5

In [ ]:
i = 4
list(dfs.keys())[i]

In [ ]:
chave = list(dfs.keys())[i]
df5 = dfs[chave]


In [ ]:
df5

In [ ]:
df5.columns

## Interpretando as Colunas

- %time e field.header.stamp: São os marcadores de tempo. O %time geralmente é o tempo de recepção do sistema, enquanto o stamp é quando o sensor gerou o dado. 

1. Coordenadas e Altitude

- field.latitude / field.longitude: Localização global do Carbon-Z. Útil se você quiser plotar o rastro do voo em um mapa para ver se ele saiu de curso após a falha.

- field.altitude: É a altitude geodésica. Como o Carbon-Z é um avião, monitorar a perda de altitude aqui é o seu principal indicador de que o motor parou de fornecer empuxo para manter o voo nivelado.

2. Confiança do Sinal (Covariância)

- field.position_covariance (0 a 8): Estas colunas formam uma matriz 3×3. Elas indicam a incerteza do GPS nos eixos X, Y e Z.

    - Valores baixos = GPS preciso.

    - Valores altos = O GPS está "perdido" ou sofrendo interferência.

- field.status.status: Indica se o GPS tem um "fix" (conexão). No dataset ALFA, você espera que seja 0 (Global Fix) ou maior.

In [ ]:
plot_generic_df(df5, titulo_grafico=primeira_chave)

## Observação:

1 - As covariâncias podem ser usadas para treinar um modelo específico para detectar falhas de sensores GPS.

2 - Há perda de altitude justamente quando o engine_status muda de 0 para 1, sinalizando uma falha do motor.

In [ ]:
# Garantir que ambos estão ordenados pelo tempo
df_merge3 = df_merge3.sort_values('%time')
df5 = df5.sort_values('%time')

# Unir os comandos de rádio ao dataframe principal
df_merge4 = pd.merge_asof(df_merge3, df5, on='%time', direction='backward')


print("Colunas agora no DF final:", df_merge4.columns.tolist())

In [ ]:
df_merge4

In [ ]:
df_merge4.columns

# Dataframe 6

In [ ]:
i = 5
list(dfs.keys())[i]

In [ ]:
chave = list(dfs.keys())[i]
df6 = dfs[chave]


In [ ]:
df6

In [ ]:
df6.columns

In [ ]:
df6 = df6.rename(columns={'field.data': 'altitude_relativa'})

In [ ]:
df6

## Interpretando as Colunas

- %time : Marcadores de tempo. O %time geralmente é o tempo de recepção do sistema. 

1. O que é o field.data neste arquivo?

Diferente da altitude do GPS (que é em relação ao nível do mar), a Relative Altitude é a altura do drone em relação ao ponto de onde ele decolou (o "Home").

- Importância: Para detectar falhas de motor em aviões, esta é a coluna mais intuitiva. Se o field.data começa a cair consistentemente enquanto o piloto mantém o acelerador alto, você tem a prova geométrica da falha.

- Dica: Renomeie essa coluna para altitude_relativa antes do merge para não confundir com a altitude global.

In [ ]:
plot_generic_df(df6, titulo_grafico=chave)

## Observação:

1 - Na análise dos primeiros gráficos, ficou claro que o avião começa a cair por volta de 116 segundos. Isso pode ser confirmado no gráfico acima.


In [ ]:
df_merge5 = pd.merge_asof(
    df_merge4, 
    df6[['%time', 'altitude_relativa']], 
    on='%time', 
    direction='backward'
)

In [ ]:
df_merge5

In [ ]:
# Limpeza de colunas duplicadas de 'header' que poluem o visual
cols_to_keep = [c for c in df_merge5.columns if not ('_x' in c or '_y' in c or 'header' in c)]
df6_final = df_merge5[cols_to_keep]

print("Colunas consolidadas no df_final:")
print(df6_final.columns.tolist())

In [ ]:
df6_final

# Dataframe 7

In [ ]:
i = 6
list(dfs.keys())[i]

In [ ]:
chave = list(dfs.keys())[i]
df7 = dfs[chave]


In [ ]:
df7

In [ ]:
df7.columns

## Interpretando as Colunas

o IMU (Inertial Measurement Unit) é, provavelmente, o arquivo mais importante para detectar uma falha de motor de forma precoce, antes mesmo do GPS notar que o avião está perdendo altitude.

Diferente do GPS, que olha para o "onde", o IMU olha para o "como" o avião está se sentindo. Aqui está a tradução técnica das colunas:

- %time e field.header.stamp: São os marcadores de tempo. O %time geralmente é o tempo de recepção do sistema, enquanto o stamp é quando o sensor gerou o dado. 

1. Orientação (Quatérnios: x, y, z, w)

Estas quatro colunas descrevem a posição do avião no espaço.

- Nota Matemática: No ROS, a orientação é dada em Quatérnios para evitar o "Gimbal Lock". Se você quiser ver graus (Roll, Pitch, Yaw), precisará convertê-los.

- Uso na Falha: Quando o motor para, o nariz do avião tende a cair (Pitch negativo) para ganhar velocidade de planeio, e as asas podem oscilar (Roll) se o piloto tentar compensar a perda de torque.

2. Velocidade Angular (angular_velocity.x, y, z)

Mede o quão rápido o avião está girando em torno de seus próprios eixos.

- Unidade: Radianos por segundo (rad/s).

- Uso na Falha: No exato momento em que o motor morre, a vibração constante do motor cessa. Você verá uma mudança no "ruído" dessas taxas.

3. Aceleração Linear (linear_acceleration.x, y, z)

Mede as forças G agindo sobre o avião.

- Fato Curioso: Em repouso no solo, o linear_acceleration.z será aproximadamente 9,81m/s2 (a gravidade da Terra).

- Uso na Falha: A coluna .x (para frente) é vital. Se o motor apaga, a aceleração positiva cessa imediatamente e o arrasto começa a desacelerar o avião (aceleração negativa).

4. Em robótica e no ROS, a covariância indica o grau de incerteza do sensor. Elas formam uma matriz 3×3 (por isso são 9 colunas, de 0 a 8).

    Valores baixos: O sensor está muito confiante de que a leitura é precisa.

    Valores altos: O dado está "sujo" ou o sensor está sofrendo interferência/vibração excessiva.

Para Cybersecurity, essas colunas são fundamentais: um ataque de injeção de dados muitas vezes esquece de "falsificar" o ruído estatístico. Se a orientação muda bruscamente, mas a covariância permanece perfeitamente estável e baixa, é um sinal de que o dado não veio de um sensor físico real, mas de um script.


Para Detecção de Falhas: O IMU é o primeiro a saber da falha. O GPS demora para atualizar a posição, mas o acelerômetro sente a perda de empuxo instantaneamente. Um modelo de IA que combina linear_acceleration.x (IMU) com rc_throttle (Controle) é extremamente robusto.

Para Segurança Cibernética: O IMU é muito difícil de falsificar perfeitamente. Se um atacante injetar dados de GPS dizendo que o avião está subindo, mas o IMU mostrar que não houve aceleração vertical (z) condizente com essa subida, temos uma prova de Ataque de Inconsistência Sensorial.

In [ ]:
plot_generic_df(df7 , titulo_grafico=chave)

# Observações:

- Todas as colunas de orientação são nulas, como podemos ver abaixo. Por isso vamos eliminá-las deste arquivo.

- A covariância é uma matriz que relaciona os três eixos (x,y,z):

### Representação da Matriz de Covariância ($3 \times 3$)

Esta matriz descreve a incerteza dos sensores nos eixos espaciais. A diagonal principal representa a variância (ruído individual), enquanto os demais campos representam a correlação entre os eixos.

| | **Eixo X** | **Eixo Y** | **Eixo Z** |
| :--- | :---: | :---: | :---: |
| **Eixo X** | **Var(X)** | Cov(X, Y) | Cov(X, Z) |
| **Eixo Y** | Cov(Y, X) | **Var(Y)** | Cov(Y, Z) |
| **Eixo Z** | Cov(Z, X) | Cov(Z, Y) | **Var(Z)** |

---


- As colunas 0, 4 e 8 (a diagonal) são as mais importantes: elas indicam a incerteza individual de cada eixo.

- As outras indicam como o erro de um eixo interfere no outro (correlação).

In [ ]:
df7['field.orientation.x'].value_counts()

In [ ]:
df7['field.orientation.y'].value_counts()

In [ ]:
df7['field.orientation.z'].value_counts()

In [ ]:
df7['field.orientation.w'].value_counts()

In [ ]:
coluns_to_drop = ['field.orientation.x', 'field.orientation.y', 'field.orientation.z', 'field.orientation.w']

df7_no_orientation = df7.drop(columns = coluns_to_drop)

In [ ]:
df7_no_orientation.columns

## Observação:

1 - As covariâncias podem ser usadas para treinar um modelo específico para detectar falhas de sensores GPS.

2 - Há perda de altitude justamente quando o engine_status muda de 0 para 1, sinalizando uma falha do motor.

In [ ]:
import pandas as pd

# 1. Garantir que os dados estejam ordenados pelo tempo (obrigatório para merge_asof)
df_merge5 = df_merge5.sort_values('%time')
df7_no_orientation = df7_no_orientation.sort_values('%time')

# 2. Realizar o merge
# Usamos 'backward' para pegar o último dado sensorial conhecido para cada instante de tempo
df_merge6 = pd.merge_asof(
    df_merge5, 
    df7_no_orientation, 
    on='%time', 
    direction='backward',
    suffixes=('', '_imu') # Adiciona sufixo se houver colunas com nomes repetidos
)

print(f"Novo formato do DataFrame: {df_merge6.shape}")

In [ ]:
df_merge6

# Dataframe 8

In [ ]:
i = 7
list(dfs.keys())[i]

In [ ]:
chave = list(dfs.keys())[i]
df8 = dfs[chave]


In [ ]:
df8

In [ ]:
df8.columns

## Interpretando as Colunas

- %time : Marcadores de tempo. O %time geralmente é o tempo de recepção do sistema. 

1. O que significam as colunas?


- field.des_x, y, z: Desired Velocity - A velocidade que o piloto automático calculou para cumprir a missão.
- field.meas_x, y, z: Measured Velocity	- A velocidade real lida pelos sensores (fusão de GPS/IMU).



Para Falha de Motor: O motor falha. O controlador do drone percebe que a velocidade está caindo e tenta compensar.

- field.des_x (Desejado) deve aumentar ou se manter alto enquanto o drone tenta recuperar velocidade.

- Ao mesmo tempo, o field.meas_x (Medido) deve cair drasticamente.

- Engenharia de feature: A diferença entre o desejado e o medido (Erro=Desejado−Medido) é um dos indicadores mais fortes de falha mecânica.

Para Cybersecurity: Este arquivo é perfeito para detectar Ataques de Injeção de Controle.

- Se o field.des_x mudar bruscamente sem que o comando de rádio (rc_throttle) tenha mudado, alguém ou algum processo malicioso está injetando setpoints falsos no navegador do drone.

- Se o field.meas_x for exatamente igual ao field.des_x por muito tempo (sem erro nenhum), é um sinal de que os dados são sintéticos, pois sensores reais sempre apresentam um pequeno erro/ruído em relação ao desejado.

In [ ]:
mapeamento_nav = {
    'field.des_x': 'vel_desejada_x',
    'field.meas_x': 'vel_medida_x',
    'field.des_y': 'vel_desejada_y',
    'field.meas_y': 'vel_medida_y',
    'field.des_z': 'vel_desejada_z',
    'field.meas_z': 'vel_medida_z'
}
df8 = df8.rename(columns=mapeamento_nav)

# 2. Calcular o erro de rastreamento (Tracking Error)
df8['erro_vel_x'] = df8['vel_desejada_x'] - df8['vel_medida_x']

In [ ]:
plot_generic_df(df8, titulo_grafico=chave)

# Observação:

- Note que as velocidades medidas ficam oscilam com um amplitude maior quando a falha ocorre (de 116 segundos em diante).

In [ ]:
cols_to_drop = [c for c in df8.columns if 'header' in c]

df_merge6= df_merge6.drop(columns=cols_to_drop)

df8 = df8.drop(columns=cols_to_drop)

df8 = df8.drop(columns = 'field.coordinate_frame')


In [ ]:
df8

In [ ]:
df_merge_final = pd.merge_asof(
    df_merge6.sort_values('%time'),
    df8.sort_values('%time'), 
    on='%time',
    direction='backward'
)

In [ ]:
df_merge_final